## 3.2 MAC 层与帧结构原理

在上一节中，我们了解了本章的学习目标和前置要求。从前两章可知，PHY 层负责将比特变成 IQ 信号在信道中传输，PHY 层本身不关心传输的内容是什么。**MAC 层**则负责组织和管理这些内容——将上层数据和控制信令封装成标准帧格式，交给 PHY 层发送。

本节学习大纲如下：

- MAC 层在协议栈中的位置与 MAC-PHY 适配
- 三种 MAC 帧类型的结构与用途
- CRC 校验与 FER 统计
- 有效吞吐量的计算方法

### 1. MAC 层与 MAC-PHY 适配

星闪 SLE 协议栈中，MAC 层位于 PHY 层之上，负责：

- **帧的组包与解包**：将上层数据和控制消息按格式封装为帧
- **链路管理**：连接建立、断开、加密配对
- **QoS 保障**：重传、功率控制、流控

MAC 层与 PHY 层之间的适配通过两个核心函数完成：

- `mac_to_iq(mac_bytes, cfg)`：将 MAC 帧字节流经 Polar 编码、QPSK/BPSK 调制、RRC 成型后输出 IQ 信号
- `iq_to_mac(iq_signal, cfg, n_bytes)`：从接收 IQ 信号中经匹配滤波、解调、SC 解码还原出 MAC 帧字节流

这种分层设计使 PHY 层可以聚焦于"如何可靠传输"，MAC 层可以聚焦于"传输什么内容"。

### 2. 三种 MAC 帧类型

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
    <tr>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:200px;">帧类型</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:350px;">用途</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:250px;">载荷特征</th>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">信令帧 (Signaling Frame)</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">传输控制信息，如 PingRequest、PingResponse、IntervalUpdateRequest 等</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">载荷极小，仅为控制字段</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">数据帧 (AsyncDataFrame)</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">传输用户业务数据，支持 MCS 自适应</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">载荷由 payload_size 决定</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">复用帧 (MuxFrame)</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">将多条信令和一段数据合并为一个物理帧</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">载荷最大，包含控制帧+数据帧</td>
    </tr>
</table>

### 3. QPSK的调制解调原理

QPSK调制原理如下图所示，QPSK相当于两个正交的BPSK相加而成。其调制原理是将基带码元分成I、Q两路，I路是原始基带码元的奇数位置码元，Q路是原始基带码元的偶数位置码元，然后两条支路分别和对应的载波相乘实现BPSK的调制，然后将两条支路相加实现QPSK的调制。

<img src="./images/QPSK1.png" width="800">

QPSK的解调原理如下图所示，QPSK信号再分为I、Q两路和对应的载波相乘，然后经过低通滤波器后进行抽样判决，相当于作两路的BPSK解调。判决之后的I、Q路码元进行合并，I路为最终码元序列的奇数位置码元，Q路为最终码元序列的偶数位置码元，恢复出原始的码元序列。

<img src="./images/QPSK2.png" width="700">

本实验中调制和解调流程如下所示：

调制：2 比特一组 → 星座映射（00→45°, 01→135°, 11→-135°, 10→-45°）→ 上采样插零 → RRC 脉冲成型 → 复基带 IQ。

解调：RRC 匹配滤波 → 下采样（每 sps 点取一个符号）→ 最近邻星座判决。

### 4. CRC 校验

CRC（Cyclic Redundancy Check，循环冗余校验）是一种根据数据产生简短固定位数校验码的信道编码技术，利用除法及余数的原理检测数据传输或保存后可能出现的错误。

**发送端**：将 MAC 载荷视为一个二进制多项式 M(x)，与预先约定的生成多项式 G(x) 进行模 2 除法，得到的余数 R(x) 即为 CRC 校验码。将 R(x) 附加在 M(x) 后面一起发送。

**接收端**：将收到的数据同样除以 G(x)，若余数为 0 则数据无误，否则说明传输过程中出现了比特错误。

例如，以 CRC-8、生成多项式 G(x)=x^8+x^2+x+1 (即 1 0000 0111) 为例：

step 1：待发送数据 M(x) = 0001 1100

step 2：数据末端加0：数据串也就是被除数，根据多项式的最高此项在后方补0，多项式最高是几次就加几个0，根据除数可知最高此项为8，也就是补8个0，此时被除数为：0001 1100 0000 0000

step 3：数据运算：将除数与被除数的最高有效位对齐， 进行按位异或操作：

<img src="./images/CRC.png" width="600">

step 4：得到余数 R(x) = 01010100

step 5：发送带有CRC校验的数据：00011100 + 01010100= 0001 1100 0101 0100


星闪 SLE 采用 **CRC-24**，生成多项式更长，检测能力更强。仿真中 `cfg.crc_len=24` 指定使用 24 位 CRC，编码函数内部自动完成 CRC 计算与附加。


### 5. FER 和有效吞吐量

**FER（Frame Error Rate，误帧率）** = 错误帧数 / 总帧数。CRC 校验失败即整帧计错。

有效吞吐量衡量单位时间内成功传输的信息比特数。对于 MCS 等级已知的链路：

$$\text{TP} = (1 - \text{FER}) \times \text{SE} \times R_s$$

其中 SE 为频谱效率（MCS7 对应 1.75 bit/symbol），Rs = 1 MHz 为符号速率。

例如 FER=0.1 时，TP = 0.9 x 1.75 x 10^6 = 1575 kbps。

### 课后练习

请根据本节课程学习内容完成以下题目进行自测。

（单选题）星闪 SLE 标准定义的 MAC 帧类型不包括：

A. 信令帧

B. 数据帧

C. 复用帧

D. 广播帧

（单选题）mac_to_iq 函数的功能是：

A. IQ 信号转 MAC 字节

B. MAC 字节转 IQ 信号

C. MAC 字节转 bits

D. bits 转 IQ 信号

（单选题）CRC 校验中，接收端通过什么条件判断数据无误：

A. 余数为 1

B. 商为 0

C. 余数为 0

D. 校验码匹配

（单选题）三种 MAC 帧类型中，哪种帧的 FER 通常最高：

A. 信令帧

B. 数据帧

C. 复用帧

D. 都一样

（单选题）MCS=7 对应的频谱效率 SE 为：

A. 0.5

B. 1.0

C. 1.5

D. 1.75

执行以下代码获取答案

In [ ]:
!cat answer/03.02_answer.txt